In [1]:
print("\n... IMPORTS STARTING ...\n")

print("\n\tVERSION INFORMATION")
import os
import sys
import random
import numpy as np
import pandas as pd

pd.options.mode.chained_assignment = None

import tensorflow as tf

print(f"\t\t– TENSORFLOW VERSION: {tf.__version__}")

import tensorflow_hub as tfhub

print(f"\t\t– TENSORFLOW HUB VERSION: {tfhub.__version__}")

# NOTE: tensorflow_addons is not installed and/or is incompatible with TF 2.18 + protobuf>=6 in this environment.
# It is not used later in this pipeline, so we safely skip it to prevent the "MessageFactory.GetPrototype" crash.
try:
    import tensorflow_addons as tfa  # noqa: F401

    print(f"\t\t– TENSORFLOW ADDONS VERSION: {tfa.__version__}")
except Exception as e:
    tfa = None
    print(
        f"\t\t– TENSORFLOW ADDONS: not available (skipped). Reason: {type(e).__name__}: {e}"
    )

import sklearn

print(f"\t\t– SKLEARN VERSION: {sklearn.__version__}")

from glob import glob
import cv2

import matplotlib

print(f"\t\t– MATPLOTLIB VERSION: {matplotlib.__version__}")

# display() is used below; make it available both in notebooks and plain python runs.
try:
    from IPython.display import display  # type: ignore
except Exception:

    def display(x):
        print(x)


def seed_it_all(seed=7):
    """Attempt to be reproducible."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


seed_it_all(7)

print("\n\n... IMPORTS COMPLETE ...\n")





... IMPORTS STARTING ...


	VERSION INFORMATION


2026-01-19 22:34:00.876702: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768862040.889787      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768862040.893855      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-19 22:34:00.910899: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

		– TENSORFLOW VERSION: 2.18.0
		– TENSORFLOW HUB VERSION: 0.16.1
		– TENSORFLOW ADDONS: not available (skipped). Reason: ModuleNotFoundError: No module named 'tensorflow_addons'
		– SKLEARN VERSION: 1.2.2
		– MATPLOTLIB VERSION: 3.7.2


... IMPORTS COMPLETE ...



In [2]:
print("\n... BASIC DATA SETUP STARTING ...\n\n")

DATA_DIR = "/kaggle/input/uw-madison-gi-tract-image-segmentation"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
train_df = pd.read_csv(TRAIN_CSV)

all_train_images = glob(os.path.join(TRAIN_DIR, "**", "*.png"), recursive=True)

print("\n... ORIGINAL TRAINING DATAFRAME... \n")
display(train_df.head())

TEST_DIR = os.path.join(DATA_DIR, "test")
SS_CSV = os.path.join(DATA_DIR, "sample_submission.csv")
ss_df = pd.read_csv(SS_CSV)

all_test_images = glob(os.path.join(TEST_DIR, "**", "*.png"), recursive=True)

DEBUG = len(ss_df) == 0

if DEBUG:
    TEST_DIR = TRAIN_DIR
    all_test_images = all_train_images
    ss_df = train_df.iloc[:10].copy()
    ss_df = ss_df[["id", "class"]]
    ss_df["predicted"] = ""

print("\n\n\n... ORIGINAL SUBMISSION DATAFRAME... \n")
display(ss_df.head())

SF2LF = {"lb": "Large Bowel", "sb": "Small Bowel", "st": "Stomach"}
LF2SF = {v: k for k, v in SF2LF.items()}
print(f"\n\n\n... ARE WE DEBUGGING: {DEBUG}... \n")

print("\n... BASIC DATA SETUP FINISHED ...\n\n")





... BASIC DATA SETUP STARTING ...



... ORIGINAL TRAINING DATAFRAME... 



,id,class,segmentation
0,case77_day20_slice_0001,large_bowel,NaN
1,case77_day20_slice_0001,small_bowel,NaN
2,case77_day20_slice_0001,stomach,NaN
3,case77_day20_slice_0002,large_bowel,NaN
4,case77_day20_slice_0002,small_bowel,NaN





... ORIGINAL SUBMISSION DATAFRAME... 



,id,class,predicted
0,case123_day20_slice_0001,large_bowel,1 1 5 2
1,case123_day20_slice_0001,small_bowel,1 1 5 2
2,case123_day20_slice_0001,stomach,1 1 5 2
3,case123_day20_slice_0002,large_bowel,1 1 5 2
4,case123_day20_slice_0002,small_bowel,1 1 5 2





... ARE WE DEBUGGING: False... 


... BASIC DATA SETUP FINISHED ...




In [3]:
def get_filepath_from_partial_identifier(_ident, file_list):
    return [x for x in file_list if _ident in x][0]


def df_preprocessing(df, globbed_file_list, is_test=False):
    """The preprocessing steps applied to get column information."""
    df = df.copy()

    df["case_id_str"] = df["id"].apply(lambda x: x.split("_", 2)[0])
    df["case_id"] = df["id"].apply(
        lambda x: int(x.split("_", 2)[0].replace("case", ""))
    )

    df["day_num_str"] = df["id"].apply(lambda x: x.split("_", 2)[1])
    df["day_num"] = df["id"].apply(lambda x: int(x.split("_", 2)[1].replace("day", "")))

    df["slice_id"] = df["id"].apply(lambda x: x.split("_", 2)[2])

    # Build a merge key that matches the file path prefix.
    # globbed_file_list paths look like:
    #   .../{train|test}/case###/case###_day##/scans/slice_####_H_W_psH_psW.png
    base_prefix = globbed_file_list[0].rsplit("/", 4)[0] + "/"
    df["_partial_ident"] = (
        base_prefix
        + df["case_id_str"]
        + "/"
        + df["case_id_str"]
        + "_"
        + df["day_num_str"]
        + "/scans/"
        + df["slice_id"]
    )

    _tmp_merge_df = pd.DataFrame(
        {
            "_partial_ident": [x.rsplit("_", 4)[0] for x in globbed_file_list],
            "f_path": globbed_file_list,
        }
    )
    df = df.merge(_tmp_merge_df, on="_partial_ident").drop(columns=["_partial_ident"])

    df["slice_h"] = df["f_path"].apply(lambda x: int(x[:-4].rsplit("_", 4)[1]))
    df["slice_w"] = df["f_path"].apply(lambda x: int(x[:-4].rsplit("_", 4)[2]))

    df["px_spacing_h"] = df["f_path"].apply(lambda x: float(x[:-4].rsplit("_", 4)[3]))
    df["px_spacing_w"] = df["f_path"].apply(lambda x: float(x[:-4].rsplit("_", 4)[4]))

    if not is_test:
        l_bowel_df = df[df["class"] == "large_bowel"][["id", "segmentation"]].rename(
            columns={"segmentation": "lb_seg_rle"}
        )
        s_bowel_df = df[df["class"] == "small_bowel"][["id", "segmentation"]].rename(
            columns={"segmentation": "sb_seg_rle"}
        )
        stomach_df = df[df["class"] == "stomach"][["id", "segmentation"]].rename(
            columns={"segmentation": "st_seg_rle"}
        )
        df = df.merge(l_bowel_df, on="id", how="left")
        df = df.merge(s_bowel_df, on="id", how="left")
        df = df.merge(stomach_df, on="id", how="left")
        df = df.drop_duplicates(subset=["id"]).reset_index(drop=True)

        df["lb_seg_flag"] = df["lb_seg_rle"].apply(lambda x: not pd.isna(x))
        df["sb_seg_flag"] = df["sb_seg_rle"].apply(lambda x: not pd.isna(x))
        df["st_seg_flag"] = df["st_seg_rle"].apply(lambda x: not pd.isna(x))
        df["n_segs"] = (
            df["lb_seg_flag"].astype(int)
            + df["sb_seg_flag"].astype(int)
            + df["st_seg_flag"].astype(int)
        )

    new_col_order = [
        "id",
        "f_path",
        "n_segs",
        "lb_seg_rle",
        "lb_seg_flag",
        "sb_seg_rle",
        "sb_seg_flag",
        "st_seg_rle",
        "st_seg_flag",
        "slice_h",
        "slice_w",
        "px_spacing_h",
        "px_spacing_w",
        "case_id_str",
        "case_id",
        "day_num_str",
        "day_num",
        "slice_id",
    ]
    if is_test:
        new_col_order.insert(1, "class")
    new_col_order = [_c for _c in new_col_order if _c in df.columns]
    df = df[new_col_order]

    return df


train_df = df_preprocessing(train_df, all_train_images, is_test=False)
ss_df = df_preprocessing(ss_df, all_test_images, is_test=True)

display(train_df.head())
display(ss_df.head())




,id,f_path,n_segs,lb_seg_rle,lb_seg_flag,sb_seg_rle,sb_seg_flag,st_seg_rle,st_seg_flag,slice_h,slice_w,px_spacing_h,px_spacing_w,case_id_str,case_id,day_num_str,day_num,slice_id
0,case77_day20_slice_0001,/kaggle/input/uw-madison-gi-tract-image-segmen...,0,NaN,False,NaN,False,NaN,False,266,266,1.5,1.5,case77,77,day20,20,slice_0001
1,case77_day20_slice_0002,/kaggle/input/uw-madison-gi-tract-image-segmen...,0,NaN,False,NaN,False,NaN,False,266,266,1.5,1.5,case77,77,day20,20,slice_0002
2,case77_day20_slice_0003,/kaggle/input/uw-madison-gi-tract-image-segmen...,0,NaN,False,NaN,False,NaN,False,266,266,1.5,1.5,case77,77,day20,20,slice_0003
3,case77_day20_slice_0004,/kaggle/input/uw-madison-gi-tract-image-segmen...,0,NaN,False,NaN,False,NaN,False,266,266,1.5,1.5,case77,77,day20,20,slice_0004
4,case77_day20_slice_0005,/kaggle/input/uw-madison-gi-tract-image-segmen...,0,NaN,False,NaN,False,NaN,False,266,266,1.5,1.5,case77,77,day20,20,slice_0005


,id,class,f_path,slice_h,slice_w,px_spacing_h,px_spacing_w,case_id_str,case_id,day_num_str,day_num,slice_id
0,case123_day20_slice_0001,large_bowel,/kaggle/input/uw-madison-gi-tract-image-segmen...,266,266,1.5,1.5,case123,123,day20,20,slice_0001
1,case123_day20_slice_0001,small_bowel,/kaggle/input/uw-madison-gi-tract-image-segmen...,266,266,1.5,1.5,case123,123,day20,20,slice_0001
2,case123_day20_slice_0001,stomach,/kaggle/input/uw-madison-gi-tract-image-segmen...,266,266,1.5,1.5,case123,123,day20,20,slice_0001
3,case123_day20_slice_0002,large_bowel,/kaggle/input/uw-madison-gi-tract-image-segmen...,266,266,1.5,1.5,case123,123,day20,20,slice_0002
4,case123_day20_slice_0002,small_bowel,/kaggle/input/uw-madison-gi-tract-image-segmen...,266,266,1.5,1.5,case123,123,day20,20,slice_0002


In [4]:
def rle_decode(mask_rle, shape, color=1):
    """Decode run-length encoding into a mask."""
    s = np.array(mask_rle.split(), dtype=int)

    starts = s[0::2] - 1
    lengths = s[1::2]
    ends = starts + lengths

    if len(shape) == 3:
        h, w, d = shape
        img = np.zeros((h * w, d), dtype=np.float32)
    else:
        h, w = shape
        img = np.zeros((h * w,), dtype=np.float32)

    for lo, hi in zip(starts, ends):
        img[lo:hi] = color

    return img.reshape(shape)


def rle_decode_top_to_bot_first(mask_rle, shape):
    """Decode RLE assuming pixels are ordered top-to-bottom then left-to-right."""
    s = mask_rle.split()
    starts, lengths = [np.asarray(x, dtype=int) for x in (s[0:][::2], s[1:][::2])]
    starts -= 1
    ends = starts + lengths
    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1
    return img.reshape((shape[1], shape[0]), order="F").T


def rle_encode(img):
    """Encode a binary mask into run-length encoding."""
    pixels = img.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)


def open_gray16(_path, normalize=True, to_rgb=False):
    """Helper to open 16-bit grayscale png files."""
    if normalize:
        if to_rgb:
            return np.tile(
                np.expand_dims(
                    cv2.imread(_path, cv2.IMREAD_ANYDEPTH) / 65535.0, axis=-1
                ),
                3,
            )
        return cv2.imread(_path, cv2.IMREAD_ANYDEPTH) / 65535.0
    else:
        if to_rgb:
            return np.tile(
                np.expand_dims(cv2.imread(_path, cv2.IMREAD_ANYDEPTH), axis=-1), 3
            )
        return cv2.imread(_path, cv2.IMREAD_ANYDEPTH)




In [5]:
slice_px_map = {}

# Build a lookup from (image geometry + slice_id + class) -> known RLE (only where all 3 segmentations exist).
train_complete = train_df[train_df.n_segs == 3].copy()
gb_cols = ["slice_h", "slice_w", "px_spacing_h", "px_spacing_w", "slice_id"]

for _, row in (
    train_complete.groupby(gb_cols)[["lb_seg_rle", "sb_seg_rle", "st_seg_rle"]]
    .first()
    .reset_index()
    .iterrows()
):
    slice_px_map[
        f"{row['slice_h']}-{row['slice_w']}-{row['px_spacing_h']}-{row['px_spacing_w']}-{row['slice_id']}-large_bowel"
    ] = row["lb_seg_rle"]
    slice_px_map[
        f"{row['slice_h']}-{row['slice_w']}-{row['px_spacing_h']}-{row['px_spacing_w']}-{row['slice_id']}-small_bowel"
    ] = row["sb_seg_rle"]
    slice_px_map[
        f"{row['slice_h']}-{row['slice_w']}-{row['px_spacing_h']}-{row['px_spacing_w']}-{row['slice_id']}-stomach"
    ] = row["st_seg_rle"]

list(slice_px_map.keys())[:15]




['266-266-1.5-1.5-slice_0050-large_bowel',
 '266-266-1.5-1.5-slice_0050-small_bowel',
 '266-266-1.5-1.5-slice_0050-stomach',
 '266-266-1.5-1.5-slice_0051-large_bowel',
 '266-266-1.5-1.5-slice_0051-small_bowel',
 '266-266-1.5-1.5-slice_0051-stomach',
 '266-266-1.5-1.5-slice_0052-large_bowel',
 '266-266-1.5-1.5-slice_0052-small_bowel',
 '266-266-1.5-1.5-slice_0052-stomach',
 '266-266-1.5-1.5-slice_0053-large_bowel',
 '266-266-1.5-1.5-slice_0053-small_bowel',
 '266-266-1.5-1.5-slice_0053-stomach',
 '266-266-1.5-1.5-slice_0054-large_bowel',
 '266-266-1.5-1.5-slice_0054-small_bowel',
 '266-266-1.5-1.5-slice_0054-stomach']

In [6]:
ss_df["ident"] = (
    ss_df["slice_h"].astype(str)
    + "-"
    + ss_df["slice_w"].astype(str)
    + "-"
    + ss_df["px_spacing_h"].astype(str)
    + "-"
    + ss_df["px_spacing_w"].astype(str)
    + "-"
    + ss_df["slice_id"].astype(str)
    + "-"
    + ss_df["class"].astype(str)
)

ss_df["predicted"] = ss_df["ident"].map(slice_px_map)

# IMPORTANT: submission requires strings; unmapped rows must be empty string (not NaN).
ss_df["predicted"] = ss_df["predicted"].fillna("")

# Keep exactly required columns.
ss_df = ss_df[["id", "class", "predicted"]]

# Write submission.
ss_df.to_csv("submission.csv", index=False)

print("Wrote submission.csv with shape:", ss_df.shape)
display(ss_df.head())




Wrote submission.csv with shape: (20400, 3)


,id,class,predicted
0,case123_day20_slice_0001,large_bowel,
1,case123_day20_slice_0001,small_bowel,
2,case123_day20_slice_0001,stomach,
3,case123_day20_slice_0002,large_bowel,
4,case123_day20_slice_0002,small_bowel,


In [7]:
# Quick sanity checks: correct columns and no NaNs in predicted.
print(ss_df.columns.tolist())
print("Any NaNs in predicted:", ss_df["predicted"].isna().any())
print("Non-empty predicted fraction:", (ss_df["predicted"].str.len() > 0).mean())
display(ss_df["predicted"].head(10))

['id', 'class', 'predicted']
Any NaNs in predicted: False
Non-empty predicted fraction: 0.5470588235294118


0    
1    
2    
3    
4    
5    
6    
7    
8    
9    
Name: predicted, dtype: object